In [ ]:
import fastf1
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from fastf1 import plotting
from src.plotset import setup_plot
from src.plotset import save_fig
from src.utils import get_acc_df

setup_plot()

In [ ]:
fastf1.Cache.enable_cache('./f1_cache')
fastf1.Cache.get_cache_info()

In [ ]:
session = fastf1.get_session(2026, 1, 'R')
session.load(weather=False, messages=False)

In [ ]:
acc_df = get_acc_df(session)

In [ ]:
# Sort drivers by 0–100 time
sorted_df = acc_df.sort_values(by='0-100')

drivers = sorted_df.index.tolist()
zero_to_hundred = sorted_df['0-100'].values
hundred_to_two = sorted_df['100-200'].values

# Compute relative deltas
delta_0_100 = zero_to_hundred - min(zero_to_hundred)
delta_100_200 = hundred_to_two - min(hundred_to_two)

gap = 0.1  # Adjust this to control the spacing between bars
y = np.arange(len(drivers))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
ax.axis('off')

for i in range(len(drivers)):
    color = plotting.get_driver_style(drivers[i],style=['color'],session=session)['color']
    ax.barh(y[i], -delta_0_100[i], left=-gap, color=color)  # Left bar
    ax.barh(y[i], delta_100_200[i], left=gap, color=color)  # Right bar

# ax.set_xlim(-1,1)
ax.invert_yaxis()  # Fastest driver at top

# Add absolute time labels on each bar, and driver name in center
for i, (driver, t1, d1, t2, d2) in enumerate(zip(drivers, zero_to_hundred, delta_0_100, hundred_to_two, delta_100_200)):
    # Left label (0–100 absolute time)
    ax.text(-d1 - 0.15, i, f'{t1:.2f}', va='center', ha='right', fontsize=16, color='w')
    # Right label (100–200 absolute time)
    ax.text(d2 + 0.15, i, f'{t2:.2f}', va='center', ha='left', fontsize=16, color='w')
    # Center driver name
    ax.text(0, i, driver, va='center', ha='center', fontsize=16, fontweight='bold', color='w')

plt.tight_layout()
plt.show()